# Goal 1 — Threats, executive responses, and business consequences
**Question.** What threats and business consequences appear in our selected CISA tabletop exercises, and what responses do the manuals ask leaders to consider?

**Scope.** The input files cover eight CISA manuals, with 66 scenario injects and 430 discussion prompts. The 16 modules or vignettes in the exercise catalogue are the unit for comparing response coverage. These are scripted exercises, so the counts describe the selected material. The MITRE example provides technical context for one explicitly described behavior.

[CISA tabletop exercise planner handbook](https://www.cisa.gov/sites/default/files/2023-01/2_-_ctep_exercise_planner_handbook_2021_final_508.pdf)

## 1. Load the datasets and check the source references
Read the classified injects, extracted prompts, and exercise catalogue. Check unique IDs and link records by manual and section. Use each row's zero-based source paragraph to verify text against the downloaded DOCX files. For injects that start at a day heading, compare the following narrative paragraphs. This checks traceability for the extracted sample.

In [1]:
%pip install reportlab


Note: you may need to restart the kernel to use updated packages.


In [2]:

from pathlib import Path
import re

import pandas as pd
from docx import Document
from reportlab.pdfgen import canvas
from reportlab.lib.colors import HexColor, white

root = next(path for path in (Path.cwd(), Path.cwd().parent)
            if (path / "data" / "processed" / "cisa_exercises.csv").is_file())
processed = root / "data" / "processed"
raw = root / "data" / "raw"
figures = root / "figures"
events = pd.read_csv(processed / "cisa_classified_events.csv")
questions = pd.read_csv(processed / "cisa_discussion_questions.csv")
exercises = pd.read_csv(processed / "cisa_exercises.csv")
keys = ["manual_title", "section"]
assert events["inject_id"].is_unique and questions["question_id"].is_unique
assert exercises["exercise_id"].is_unique and not exercises.duplicated(keys).any()

manual_files = {
    "Healthcare": "healthcare_and_public_health.docx",
    "Vendor": "vendor_supply_chain_compromise.docx",
    "Ransomware": "ransomware.docx",
    "Insider Threat": "insider_threat.docx",
    "Executives": "executives_and_senior_leadership.docx",
    "Industrial Control Systems": "industrial_control_systems.docx",
    "Financial Services": "financial_services.docx",
    "Communications": "communications.docx",
}
paragraphs = {
    manual: [paragraph.text.strip() for paragraph in Document(raw / filename).paragraphs]
    for manual, filename in manual_files.items()
}

def normalized(text):
    return " ".join(text.split())

for row in questions.itertuples():
    assert normalized(row.question_text) == normalized(
        paragraphs[row.manual_title][row.source_paragraph]), row.question_id
for row in events.itertuples():
    source = paragraphs[row.manual_title][row.source_paragraph:]
    if re.match(r"^Day\s+\d+", source[0], re.I):
        source = source[1:]
    narrative = normalized(row.event_description)
    source_prefix = normalized(" ".join(source[:12]))
    assert source_prefix == narrative or source_prefix.startswith(narrative + " "), row.inject_id

events = events.merge(exercises[keys + ["exercise_id"]], on=keys, validate="many_to_one", how="left")
questions = questions.merge(exercises[keys + ["exercise_id"]], on=keys, validate="many_to_one", how="left")
assert events["exercise_id"].notna().all()
general = questions[questions["exercise_id"].isna()]
assert general["section"].eq("General Discussion Questions").all()
print(f"Verified {len(events)} injects and {len(questions)} prompts in {len(manual_files)} manuals.")
print(f"{questions['exercise_id'].count()} prompts are linked to {len(exercises)} exercises.")
print(f"{len(general)} general prompts are outside the exercise-level comparison.")

Verified 66 injects and 430 prompts in 8 manuals.
410 prompts are linked to 16 exercises.
20 general prompts are outside the exercise-level comparison.


## 2. Check the threat and impact labels
Keep one **primary** threat and impact per inject so every heatmap cell represents an inject count. A primary threat is a narrative category. For example, an inject can describe a vulnerability or warning without reporting a completed attack.

Classify the primary consequence relative to the **organization participating in the exercise**:

- **Described as occurring** — the narrative reports it within the exercise, including concrete effects such as disruption or stakeholder reactions.
- **Potential or unconfirmed** — the effect is threatened, suspected, prospective, or its magnitude is unconfirmed.
- **None reported** — the inject describes no new direct consequence for the organization.

Preserve the original fields for comparison. The review notes below record cases where the source changes how the original annotation should be interpreted. Secondary consequences can appear in the notes and examples. They do not add to the heatmap totals.

In [3]:
events["reviewed_threat"] = events["threat_category"]
events["reviewed_impact"] = events["impact_category"]
events["impact_status"] = events["impact_category"].map({
    "No Direct Impact": "None reported",
    "Security Risk": "Potential or unconfirmed",
}).fillna("Described as occurring")
events["review_note"] = ""

# Entries record source-specific judgments about the primary consequence.
reviews = {
    "HC-INJ-137": ("Patient Safety", "Potential or unconfirmed",
                   "Incorrect records are reported. Patient harm is not described."),
    "HC-INJ-139": ("Patient Safety", "Potential or unconfirmed",
                   "Incorrect infusion delivery is reported. Patient harm is not described."),
    "VD-INJ-111": ("No Direct Impact", "None reported",
                   "Other organizations experience disruptions. The participating organization is unaffected at this point."),
    "VD-INJ-136": ("Security Risk", "Potential or unconfirmed",
                   "Sector partners' data appears online. Exposure of the participating organization's data is not reported."),
    "RW-INJ-122": ("Data Exposure", "Potential or unconfirmed",
                   "The outbound transfer is described. The contents and sensitivity of the data are unspecified."),
    "RW-INJ-124": ("Operational Disruption", "Described as occurring",
                   "Screens are interrupted and display a ransom demand. The extent of disruption is unspecified."),
    "RW-INJ-126": ("Data Exposure", "Described as occurring",
                   "Some records are shared as evidence. Wider exposure and sale remain uncertain."),
    "RW-INJ-128": ("Reputational Damage", "Potential or unconfirmed",
                   "News coverage and requests for comment are described. Reputational damage is a possible consequence."),
    "IT-INJ-87": ("Financial Cost", "Potential or unconfirmed",
                  "The narrative cites expense for the planned replacement without reporting realized costs."),
    "IT-INJ-91": ("Personnel Consequences", "Described as occurring",
                  "Workplace conflict occurs. The termination is described in the following inject."),
    "IT-INJ-93": ("Personnel Consequences", "Described as occurring",
                  "Management terminates the employee. Retaliation is only a threat."),
    "IT-INJ-121": ("Operational Disruption", "Described as occurring",
                   "Replacement is underway and constrained by material supply. The 18-month duration is an estimate."),
    "IT-INJ-127": ("Security Risk", "Described as occurring",
                   "An unauthorized account is observed and disabled. Further compromise is uncertain."),
    "IT-INJ-129": ("Security Risk", "Described as occurring",
                   "The scenario describes compromised credentials and unauthorized privileged access."),
    "IT-INJ-131": ("Security Risk", "Described as occurring",
                   "Unauthorized account creation is established. The vendor-associated address does not establish culpability."),
    "IT-INJ-133": ("Operational Disruption", "Described as occurring",
                   "Critical files are unavailable to staff. Permanent deletion is suspected."),
    "ICS-INJ-165": ("Operational Disruption", "Potential or unconfirmed",
                    "The actor threatens to continue the lockout. The earlier shutdown is counted in ICS-INJ-146."),
    "FS-INJ-141": ("Reputational Damage", "Potential or unconfirmed",
                   "Media requests are reported. Damage to reputation is a potential consequence."),
    "FS-INJ-150": ("Reputational Damage", "Described as occurring",
                   "Widespread attempts to withdraw or close accounts show a customer reaction. Completed withdrawals are not stated."),
    "CM-INJ-141": ("Reputational Damage", "Described as occurring",
                   "Customers publicly express concern. Long-term reputational effects are unspecified."),
}
assert set(reviews).issubset(set(events["inject_id"]))
events = events.set_index("inject_id", drop=False)
for inject_id, note in reviews.items():
    events.loc[inject_id, ["reviewed_impact", "impact_status", "review_note"]] = note

# Separate the personnel action from the possibility of a later cyber incident.
events.loc["IT-INJ-93", "reviewed_threat"] = "No Confirmed Incident"
status_counts = events["impact_status"].value_counts()
print(status_counts.to_string())
display(events.loc[list(reviews), ["threat_category", "reviewed_threat",
        "impact_category", "reviewed_impact", "impact_status", "review_note"]].reset_index(drop=True)
        .assign(inject_id=list(reviews)).set_index("inject_id"))

impact_status
Described as occurring      31
Potential or unconfirmed    23
None reported               12


,threat_category,reviewed_threat,impact_category,reviewed_impact,impact_status,review_note
inject_id,,,,,,
HC-INJ-137,Other Security Incident,Other Security Incident,Patient Safety,Patient Safety,Potential or unconfirmed,Incorrect records are reported. Patient harm i...
HC-INJ-139,System Disruption,System Disruption,Patient Safety,Patient Safety,Potential or unconfirmed,Incorrect infusion delivery is reported. Patie...
VD-INJ-111,Third-Party / Supply Chain,Third-Party / Supply Chain,No Direct Impact,No Direct Impact,None reported,Other organizations experience disruptions. Th...
VD-INJ-136,Third-Party / Supply Chain,Third-Party / Supply Chain,Data Exposure,Security Risk,Potential or unconfirmed,Sector partners' data appears online. Exposure...
RW-INJ-122,Data Exposure,Data Exposure,Data Exposure,Data Exposure,Potential or unconfirmed,The outbound transfer is described. The conten...
RW-INJ-124,Ransomware,Ransomware,Operational Disruption,Operational Disruption,Described as occurring,Screens are interrupted and display a ransom d...
RW-INJ-126,Data Exposure,Data Exposure,Data Exposure,Data Exposure,Described as occurring,Some records are shared as evidence. Wider exp...
RW-INJ-128,Ransomware,Ransomware,Reputational Damage,Reputational Damage,Potential or unconfirmed,News coverage and requests for comment are des...
IT-INJ-87,Vulnerability,Vulnerability,Financial Cost,Financial Cost,Potential or unconfirmed,The narrative cites expense for the planned re...


## 3. Count threats and impacts
Cross-tabulate the reviewed labels and check that the counts sum to the number of injects. The first table includes all injects. The second limits the input to consequences described as occurring, so a threat or possible consequence is distinguishable from a reported effect within the exercise.

In [4]:
threats = events["reviewed_threat"].value_counts()
impacts = events["reviewed_impact"].value_counts()
matrix = pd.crosstab(events["reviewed_threat"], events["reviewed_impact"]).reindex(
    index=threats.index, columns=impacts.index, fill_value=0)
occurring = events[events["impact_status"].eq("Described as occurring")]
occurring_matrix = pd.crosstab(occurring["reviewed_threat"], occurring["reviewed_impact"])
assert matrix.to_numpy().sum() == len(events)
assert occurring_matrix.to_numpy().sum() == len(occurring)
display(matrix)
display(occurring_matrix)

reviewed_impact,Security Risk,Operational Disruption,No Direct Impact,Data Exposure,Reputational Damage,Patient Safety,Personnel Consequences,Financial Cost,Data Loss
reviewed_threat,,,,,,,,,
No Confirmed Incident,0,0,10,0,0,0,2,0,0
Ransomware,0,7,0,0,3,0,0,0,0
Insider Threat / Unauthorized Access,5,1,0,1,0,0,0,0,1
Data Exposure,0,1,0,6,1,0,0,0,0
System Disruption,0,5,0,0,0,1,0,0,0
Phishing,5,0,0,0,0,0,0,0,0
Vulnerability,3,1,0,0,0,0,0,1,0
Third-Party / Supply Chain,4,0,1,0,0,0,0,0,0
Denial of Service,0,2,1,0,1,0,0,0,0


reviewed_impact,Data Exposure,Data Loss,Operational Disruption,Personnel Consequences,Reputational Damage,Security Risk
reviewed_threat,,,,,,
Data Exposure,5,0,1,0,1,0
Denial of Service,0,0,2,0,1,0
Insider Threat / Unauthorized Access,1,1,1,0,0,2
No Confirmed Incident,0,0,0,2,0,0
Ransomware,0,0,6,0,1,0
System Disruption,0,0,5,0,0,0
Third-Party / Supply Chain,0,0,0,0,0,1
Vulnerability,0,0,1,0,0,0


## 4. Code the executive response prompts
Use explicit phrases to flag prompts about escalation, maintaining operations, restoration, stakeholder communication, legal notifications, and executive risk decisions. A question can match multiple themes. Inspect matching questions by ID and keep unmatched questions visible. This is a conservative text-based analysis of six selected themes. Broader planning, staff training, and technical readiness prompts can fall outside these themes.

Count a theme once per exercise if at least one linked prompt matches it. The 20 general questions are excluded from that comparison because they are outside the 16 catalogue items. A prompt match means the topic is raised for discussion, even if it asks about an existing process.

In [5]:
theme_rules = {
    "Escalation and activation": (
        r"escalat|(?=.*(?:activat|authoriz))(?=.*(?:incident response|continuity|\bcirp\b|\bbcp\b|\bcoop\b))"
        r"|severity.*(?:assess|dictate|determine)|assess.*severity|at what point.*notif"),
    "Business continuity": (
        r"continuity|continuing operations|continue (?:to provide|operations|essential)"
        r"|maintain(?:ing)? (?:essential|operations)|alternate (?:system|procedure)|alternative systems|manual processes"
        r"|redundant systems|manual mode"),
    "Restoration and backups": (
        r"restor(?:e|ation|ing)(?! efforts)|\bbackups?\b|\bbacked[- ]up\b|recovery (?:time|point)"),
    "Stakeholder communication": (
        r"public messaging|crisis communication|pre[- ]scripted messages|media (?:reports|inquiries)"
        r"|respond to the media|communicat\w+ with|message to|public.s confidence|inform (?:your )?customers|are you sharing|messages out to"),
    "Legal and regulatory response": (
        r"legal counsel|legal and regulatory|notification laws|legal.*ramifications"
        r"|reporting requirements|regulatory (?:requirements|notifications)|breach notification"),
    "Executive risk decisions": (
        r"ransom(?:ware)? payment|(?:refus\w*|agree\w*|decid\w*).*\bpay\b"
        r"|paying or not paying|advantages and disadvantages|trade[- ]offs"),
}
linked = questions[questions["exercise_id"].notna()].copy()
flags = pd.DataFrame({
    theme: linked["question_text"].str.contains(pattern, case=False, regex=True)
    for theme, pattern in theme_rules.items()
}, index=linked.index)
prompt_codes = linked[["exercise_id", "question_id", "question_text"]].join(flags)
by_exercise = prompt_codes.groupby("exercise_id")[list(theme_rules)].any().reindex(exercises["exercise_id"])
assert by_exercise.notna().all().all()
response_counts = pd.DataFrame({
    "Exercises": by_exercise.sum(),
    "Matching prompts": flags.sum(),
}).sort_values("Exercises", ascending=False)
print(f"{flags.any(axis=1).sum()} of {len(linked)} linked prompts match at least one rule.")
display(response_counts)
review_examples = []
for theme in theme_rules:
    examples = prompt_codes.loc[prompt_codes[theme], ["exercise_id", "question_id", "question_text"]].head(2)
    review_examples.append(examples.assign(theme=theme))
display(pd.concat(review_examples, ignore_index=True))
display(prompt_codes.loc[~flags.any(axis=1), ["exercise_id", "question_id", "question_text"]].head(6))

102 of 410 linked prompts match at least one rule.


,Exercises,Matching prompts
Business continuity,9,33
Escalation and activation,8,18
Restoration and backups,7,21
Stakeholder communication,6,20
Legal and regulatory response,5,7
Executive risk decisions,3,7


,exercise_id,question_id,question_text,theme
0,EX-02,HC-Q-144,What guidance does the plan include on assessi...,Escalation and activation
1,EX-02,HC-Q-145,How does incident severity level dictate respo...,Escalation and activation
2,EX-02,HC-Q-151,What redundant systems exist for when primary ...,Business continuity
3,EX-02,HC-Q-152,What alternative systems or manual processes a...,Business continuity
4,EX-02,HC-Q-147,Does your organization have backups of vital r...,Restoration and backups
5,EX-02,HC-Q-148,How frequently do you run backups?,Restoration and backups
6,EX-02,HC-Q-173,Describe your organizational processes to resp...,Stakeholder communication
7,EX-02,HC-Q-174,What pre-scripted messages have been developed...,Stakeholder communication
8,EX-02,HC-Q-170,Discuss potential legal and reputational ramif...,Legal and regulatory response
9,EX-02,HC-Q-172,What security breach notification laws does yo...,Legal and regulatory response


,exercise_id,question_id,question_text
0,EX-01,HC-Q-93,What are the greatest cyber threats to your or...
1,EX-01,HC-Q-94,What cybersecurity threat information does you...
2,EX-01,HC-Q-95,What cyber threat information is most useful?
3,EX-01,HC-Q-96,How is information disseminated throughout you...
4,EX-01,HC-Q-97,What actions would your organization take in r...
5,EX-01,HC-Q-98,Has your organization conducted a risk assessm...


## 5. Export two poster figures
Save a vector PDF heatmap of primary threat and impact counts and a vector PDF bar chart of response theme coverage. The heatmap includes potential consequences and injects with no direct impact. The response chart counts distinct exercises.

In [6]:
figures.mkdir(exist_ok=True)
ink = HexColor("#1b263b")
muted = HexColor("#45566a")

def write_text(chart, x, y, text, size=11, color=ink, bold=False, align="left"):
    chart.setFillColor(color)
    chart.setFont("Helvetica-Bold" if bold else "Helvetica", size)
    if align == "right":
        chart.drawRightString(x, y, str(text))
    elif align == "center":
        chart.drawCentredString(x, y, str(text))
    else:
        chart.drawString(x, y, str(text))

heatmap_path = figures / "goal1_threat_impact.pdf"
chart = canvas.Canvas(str(heatmap_path), pagesize=(860, 510))
chart.setTitle("Primary threats and impacts in the CISA exercises")
write_text(chart, 26, 476, "Primary threats and impacts in the CISA exercises", 23, bold=True)
write_text(chart, 26, 450, f"{len(events)} scripted injects across {len(manual_files)} manuals | One primary pair per inject", 12, muted)
left, top, cell_width, row_height = 248, 414, 62, 29
palette = ["#f0f4f8", "#d8e9f0", "#a9cfdd", "#78afc4", "#508ba8",
           "#346e91", "#285571", "#1e425c", "#15354b"]
maximum = int(matrix.to_numpy().max())
for y, (threat, row) in enumerate(matrix.iterrows()):
    bottom = top - (y + 1) * row_height
    write_text(chart, left - 9, bottom + 10, threat, align="right")
    for x, value in enumerate(row):
        shade = round(8 * value / maximum)
        chart.setFillColor(HexColor(palette[shade]))
        chart.rect(left + x * cell_width, bottom, cell_width - 2, row_height - 2, fill=1, stroke=0)
        write_text(chart, left + (x + 0.5) * cell_width, bottom + 9,
                   value, color=white if shade >= 4 else ink, align="center")
for x, label in enumerate(matrix.columns):
    chart.saveState()
    chart.translate(left + (x + 0.5) * cell_width + 4, top - len(matrix) * row_height - 7)
    chart.rotate(48)
    write_text(chart, 0, 0, label, size=10, align="right")
    chart.restoreState()
write_text(chart, 26, 29, "Includes potential consequences and injects with no direct impact. Values are counts.", 11, muted)
chart.save()

response_path = figures / "goal1_executive_response_themes.pdf"
chart = canvas.Canvas(str(response_path), pagesize=(770, 364))
chart.setTitle("Executive response themes across CISA exercises")
write_text(chart, 24, 331, "Executive response themes across CISA exercises", 22, bold=True)
write_text(chart, 24, 307, "Count of modules or vignettes with at least one matching prompt", 12, muted)
left, right, bottom = 244, 710, 66
for tick in range(0, len(exercises) + 1, 4):
    x = left + (right - left) * tick / len(exercises)
    chart.setStrokeColor(HexColor("#dce4eb"))
    chart.line(x, bottom, x, 283)
    write_text(chart, x, bottom - 15, tick, size=10, align="center")
for index, (theme, value) in enumerate(response_counts["Exercises"].items()):
    y = 267 - index * 34
    width = (right - left) * value / len(exercises)
    write_text(chart, left - 10, y, theme, size=12, align="right")
    chart.setFillColor(HexColor("#246d86"))
    chart.rect(left, y - 4, width, 18, fill=1, stroke=0)
    write_text(chart, left + width + 8, y, value, bold=True)
write_text(chart, (left + right) / 2, 33, f"Exercises (out of {len(exercises)})", 11, muted, align="center")
chart.save()
print(heatmap_path.relative_to(root))
print(response_path.relative_to(root))

figures/goal1_threat_impact.pdf
figures/goal1_executive_response_themes.pdf


## 6. Walk through Ransomware Module 2 and check one MITRE link
Read the events and a selection of questions for EX-06. The scenario moves from an unusual outbound data transfer to a ransom demand, a report that some records were shared, and media attention. The response prompts ask leaders to prioritize restoration, consider alternatives for keeping services running, weigh the payment decision, and coordinate public messaging. They are prompts for discussion at the module level.

For EX-06, an actor's demand for payment for a decryption key is consistent with **T1486 (Data Encrypted for Impact)**. The text does not independently confirm the encryption, so leave that link provisional. **ICS-INJ-146** explicitly describes encrypted control data and interrupted availability. That behavior supports a match to T1486. The local ATT&CK export connects T1486 to **M1053 (Data Backup)**, which provides technical context for the continuity and restoration discussion. The scenario does not report whether restoration succeeds.

[MITRE T1486](https://attack.mitre.org/techniques/T1486/) · [MITRE M1053](https://attack.mitre.org/mitigations/M1053/)

In [7]:
case = events[events["exercise_id"].eq("EX-06")]
display(case[["source_paragraph", "reviewed_threat", "reviewed_impact", "impact_status"]])
case_prompt_ids = ["RW-Q-133", "RW-Q-143", "RW-Q-150", "RW-Q-160"]
case_prompts = linked[linked["question_id"].isin(case_prompt_ids) & linked["exercise_id"].eq("EX-06")]
assert len(case_prompts) == len(case_prompt_ids)
display(case_prompts[["question_id", "source_paragraph", "question_text"]].set_index("question_id"))

techniques = pd.read_csv(processed / "mitre_techniques.csv")
mitigations = pd.read_csv(processed / "technique_mitigations.csv")
technique = techniques.loc[techniques["technique_id"].eq("T1486"), ["technique_id", "name"]]
backup = mitigations.loc[
    mitigations["technique_id"].eq("T1486") & mitigations["mitigation_id"].eq("M1053"),
    ["technique_id", "mitigation_id", "mitigation_name"]]
assert len(technique) == 1 and len(backup) == 1
assert "data is encrypted" in events.loc["ICS-INJ-146", "event_description"].lower()
display(technique)
display(backup)

,source_paragraph,reviewed_threat,reviewed_impact,impact_status
inject_id,,,,
RW-INJ-122,122,Data Exposure,Data Exposure,Potential or unconfirmed
RW-INJ-124,124,Ransomware,Operational Disruption,Described as occurring
RW-INJ-126,126,Data Exposure,Data Exposure,Described as occurring
RW-INJ-128,128,Ransomware,Reputational Damage,Potential or unconfirmed


,source_paragraph,question_text
question_id,,
RW-Q-133,133,How has your organization prioritized IT infra...
RW-Q-143,143,What alternative systems or manual processes a...
RW-Q-150,150,Discuss the advantages and disadvantages of ei...
RW-Q-160,160,How would public messaging be coordinated and ...


,technique_id,name
490,T1486,Data Encrypted for Impact


,technique_id,mitigation_id,mitigation_name
1434,T1486,M1053,Data Backup


## 7. Interpret the results
The counts below describe the material in this catalogue. Several injects can describe successive stages of the same incident. The theme counts depend on the selected phrases and the amount of discussion material in each exercise. Read the examples alongside the counts, particularly where a theme might be expressed using different language. The findings support exercise selection and design within this catalogue.

In [8]:
top_threat = threats.drop("No Confirmed Incident").idxmax()
top_impact = occurring["reviewed_impact"].value_counts().idxmax()
theme = response_counts.index[0]
print(f"{top_threat} is the most common primary threat category among injects "
      f"with a specific category ({threats[top_threat]} of {len(events)} total injects).")
print(f"{top_impact} is the most common primary consequence described as occurring "
      f"({(occurring['reviewed_impact'] == top_impact).sum()} of {len(occurring)} injects in that group).")
print(f"{theme} has the widest coverage under the six prompt rules "
      f"({response_counts.loc[theme, 'Exercises']} of {len(exercises)} exercises).")
print(f"For {top_threat.lower()}, {matrix.loc[top_threat, 'Operational Disruption']} of "
      f"{threats[top_threat]} injects have operational disruption as the primary impact. "
      "The first heatmap includes both reported and potential impacts.")

Ransomware is the most common primary threat category among injects with a specific category (10 of 66 total injects).
Operational Disruption is the most common primary consequence described as occurring (16 of 31 injects in that group).
Business continuity has the widest coverage under the six prompt rules (9 of 16 exercises).
For ransomware, 7 of 10 injects have operational disruption as the primary impact. The first heatmap includes both reported and potential impacts.


In [9]:
reviewed_columns = ["inject_id", "reviewed_threat", "reviewed_impact", "impact_status", "review_note"]
events[reviewed_columns].to_csv(processed / "cisa_events_reviewed.csv", index=False, encoding="utf-8-sig")
print("Saved cisa_events_reviewed.csv:", len(events), "rows")

Saved cisa_events_reviewed.csv: 66 rows
